This is for collecting local llm results and calculate metrics

1. download results and save them into folder `localllm_results`. follow the structure of the folders
2. `python gather_localllm_results.py ...` set the correct parameters 
3. merge prediction with ground truth using the following code
4. use SDMBench `calculate.ipynb` to calculate metrics
5. plot 

In [ ]:
from src.paths import dataset_dir, dataset_file, dataset_root, repository_root


In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.model_selection import train_test_split
from src.utils import * 
import src.prompt as prompt
from src.data_loader import load_spatial_data_csv, load_spatial_data_anndata


In [ ]:
config = load_config("configs/config_zeroshot_libd.yaml")
config.data_name = "151509"
# --- Load data ---
data_path = str(dataset_dir("visium_libd", config.data_name))

adata = load_spatial_data_anndata(
    data_path=data_path,
    adata_file="filtered_feature_bc_matrix.h5",
    config=config,
    celltype_path=os.path.join("examples/visium_libd/deconv_result", f"celltype_proportions_{config.data_name}.csv"),
    optional_files=["metadata.tsv"]
)

In [ ]:
truth_df = adata.obs
truth_df.to_csv(f"examples/intermediates/local_llm_results/LIBD/{config.data_name}_all_ground_truth.csv")

local_LLM_results = pd.read_csv(f"examples/intermediates/local_llm_results/processed/ttrlgc/{config.data_name}_zeroshot_all_results.csv")

In [ ]:
local_LLM_all_in_one_reaults = pd.DataFrame()
data_types = ["libd"]
data_names = ["151509_zeroshot"]  # , "BZ9_all", "BZ14_all"
model_names = local_LLM_results.model.unique() 
location_key = config.pos_name
truth_key = config.name_truth
reps = ["rep1", "rep2", "rep3"]  # , "rep2", "rep3"
experiment_types = local_LLM_results.experiment_type.unique()

for data_type in data_types:
    for data in data_names:
        for model in model_names:
            for experiment_type in experiment_types:
                for rep in reps:
                    # if data == "151673_test":
                    #     obs_df = ground_truth_151673.copy()
                    #     local_llm_results_df = local_LLM_results_151673.copy()
                    # elif data == "151674_all":
                    #     obs_df = ground_truth_151674.copy()
                    #     local_llm_results_df = local_LLM_results_151674.copy()
                    # elif data == "151675_all":
                    #     obs_df = ground_truth_151675.copy()
                    #     local_llm_results_df = local_LLM_results_151675.copy()
                    # elif data == "151676_all":
                    #     obs_df = ground_truth_151676.copy()
                    #     local_llm_results_df = local_LLM_results_151676.copy()
                    obs_df = truth_df.copy()
                    local_llm_results_df = local_LLM_results.copy()
                    setting_results = local_llm_results_df[
                        (local_llm_results_df['model'] == model) & 
                        (local_llm_results_df['experiment_type'] == experiment_type) & 
                        (local_llm_results_df['data_name'] == data) &
                        (local_llm_results_df['replicate'] == rep)
                    ].copy()
                    if len(setting_results) == 0:
                        continue
                    #print(obs_df.columns)
                    setting_results.index = obs_df.index

                    # drop useless columns in obs_df
                    obs_df = obs_df.drop(columns=['replicate'])


                    obs_df = obs_df.join(setting_results, how='left')

                    # refine results
                    adj_matrix, _ = sparse_adjacency(obs_df[location_key], threshold=config.r)
                    refined_niche = relabel_cells(adj_matrix.toarray(), obs_df['prediction'])
                    obs_df['local_llm_result_refined'] = refined_niche

                    local_LLM_all_in_one_reaults = pd.concat([local_LLM_all_in_one_reaults, obs_df])




In [ ]:
print(local_LLM_all_in_one_reaults.model.unique())
print(local_LLM_all_in_one_reaults.experiment_type.unique())
print(x.data_name.unique())
print(local_LLM_all_in_one_reaults.replicate.unique())


In [ ]:
# save previous results as old results
old_local_LLM_all_in_one_reaults = pd.read_csv("examples/results/localllm_libd_all_in_one_results.csv", index_col=0)
old_local_LLM_all_in_one_reaults.to_csv("examples/results/old_localllm_libd_all_in_one_results.csv")

In [ ]:
x = pd.concat([old_local_LLM_all_in_one_reaults, local_LLM_all_in_one_reaults])

In [ ]:
x.to_csv("examples/results/localllm_libd_all_in_one_results.csv")

# plot
code is from `plot_compare_different_llm.ipynb`

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['pdf.fonttype'] = 42

In [ ]:
# zeroshot
original_results_df = pd.read_csv('examples/results/zeroshot_all_metrics_all_replicates.csv', index_col=0)
original_results_df = original_results_df[original_results_df['data_type'] == 'visium']
original_results_df = original_results_df[original_results_df['model_name'] == 'gpt4o_mini']
original_results_df['experiment_types'] = 'zeroshot'

# finetune
finetune_results_df = pd.read_csv('examples/results/finetunePro_all_metrics_all_replicates.csv', index_col=0)
finetune_results_df = finetune_results_df[finetune_results_df['data_type'] == 'visium']
finetune_results_df = finetune_results_df[finetune_results_df['stage'] == 'one']
finetune_results_df = finetune_results_df[finetune_results_df['finetune_name'] == '151673']
finetune_results_df['experiment_types'] = 'finetune_151673'
finetune_results_df['model_name'] = 'gpt4o_mini'

# local LLM
local_LLM_all_metrics = pd.read_csv("examples/results/localllm_libd_all_metrics.csv", index_col=0)
local_LLM_all_metrics['data_name'] = local_LLM_all_metrics['data_name'].str.split('_').str[0]
local_LLM_all_metrics['data_type'] = 'visium'


In [ ]:
column_to_compare = ['data_name', 'model_name', 'experiment_types', 'replicate',
       'ARI', 'NMI', 'HOM', 'COM', 'CHAOS', 'PAS', 'ASW']
all_results_df = pd.concat([original_results_df[column_to_compare].copy(), 
                            finetune_results_df[column_to_compare].copy(), 
                            local_LLM_all_metrics[column_to_compare].copy()])

In [ ]:
all_results_df

In [ ]:
# Example 1: Plot ARI for zeroshot experiment
fig, ax = plot_metric_comparison(all_results_df, 
                                 metric='NMI', 
                                 experiment_type='zeroshot',
                                 data_names=['151509'],
                                 model_names = [ "Qwen2.5-7", "gpt4o_mini"],
                                 figsize=(10, 6),
                                 ylim=(0, 1))
plt.show()


In [ ]:

def plot_boxplot_comparison_by_dataset(all_results_df, metric, 
                                       data_names=['BZ5', 'BZ9', 'BZ14'],
                                       baseline_models=None, compare_models=None,
                                       baseline_exp='zeroshot', compare_exp='finetune_BZ5',
                                       figsize=(15, 5), palette=None, ylim=None, show_points=True):
    """
    Plot boxplot comparing two experiment types (baseline vs comparison) across different models with pair-to-pair comparison.
    Creates separate subplots for each dataset (data_name).
    
    Parameters:
    -----------
    all_results_df : pd.DataFrame
        DataFrame containing columns: data_name, model_name, experiment_types, and metric columns
    metric : str
        The metric to plot (e.g., 'ARI', 'NMI', 'HOM', etc.)
    data_names : list, optional
        List of data names to plot.
        If None, uses default: ['BZ5', 'BZ9', 'BZ14']
    baseline_models : list, optional
        List of model names to use for baseline experiment.
        If None, uses default: ['Llama8', 'Qwen30', 'Llama70', 'gpt4o_mini']
    compare_models : list, optional
        List of model names to use for comparison experiment. Should have same length as baseline_models.
        If None, uses the same models as baseline_models.
    baseline_exp : str, optional
        The baseline experiment type name in the dataframe (default: 'zeroshot')
    compare_exp : str, optional
        The comparison experiment type name in the dataframe (default: 'finetune_BZ5')
    figsize : tuple, optional
        Figure size (width, height)
    palette : dict or str, optional
        Color palette for experiment types. If dict, keys should be experiment type names (values of baseline_exp and compare_exp).
    ylim : tuple, optional
        Y-axis limits (min, max)
    show_points : bool, optional
        Whether to show individual data points on top of boxes
    
    Returns:
    --------
    fig, axes : matplotlib figure and axis objects
    """
    # Set default model lists if not provided
    if baseline_models is None:
        baseline_models = ['Llama8', 'Qwen30', 'Llama70', 'gpt4o_mini']
    
    if compare_models is None:
        compare_models = baseline_models.copy()
    
    # Validate that both lists have the same length
    if len(baseline_models) != len(compare_models):
        raise ValueError(f"baseline_models and compare_models must have the same length. "
                        f"Got {len(baseline_models)} and {len(compare_models)} respectively.")
    
    # Filter data for the two experiment types
    filtered_df = all_results_df[
        all_results_df['experiment_types'].isin([baseline_exp, compare_exp])
    ].copy()
    
    if filtered_df.empty:
        print(f"No data found for experiment types: {baseline_exp} or {compare_exp}")
        return None, None
    
    # Get unique datasets
    n_datasets = len(data_names)
    
    # Create subplots
    fig, axes = plt.subplots(1, n_datasets, figsize=figsize, sharey=True)
    if n_datasets == 1:
        axes = [axes]
    
    # Define colors
    if palette is None:
        palette = {
            baseline_exp: '#8dd3c7',
            compare_exp: '#fb8072'
        }
    
    # Plot each dataset
    for idx, data_name in enumerate(data_names):
        ax = axes[idx]
        
        # Filter for this dataset
        dataset_df = filtered_df[filtered_df['data_name'] == data_name].copy()
        
        # Check if data exists for this dataset
        if dataset_df.empty:
            continue

        # Create a combined column for x-axis grouping
        dataset_df['model_exp'] = dataset_df['model_name'] + '\n' + dataset_df['experiment_types']
        
        # Define the order for x-axis (alternating baseline and comparison for each model pair)
        x_order = []
        for b_model, c_model in zip(baseline_models, compare_models):
            x_order.append(f"{b_model}\n{baseline_exp}")
            x_order.append(f"{c_model}\n{compare_exp}")
        
        # Filter to only include models that exist in the data
        x_order = [x for x in x_order if x in dataset_df['model_exp'].values]
        
        # Create color list for each box based on experiment type
        colors = []
        for x in x_order:
            if baseline_exp in x:
                colors.append(palette.get(baseline_exp, '#8dd3c7'))
            else:
                colors.append(palette.get(compare_exp, '#fb8072'))
        
        if not x_order:
            continue
            
        # Create boxplot
        box_data = [dataset_df[dataset_df['model_exp'] == x][metric].values for x in x_order]
        
        bp = ax.boxplot(box_data, 
                        labels=x_order,
                        patch_artist=True,
                        widths=0.6,
                        showmeans=True,
                        meanprops=dict(marker='D', markerfacecolor='red', markersize=6, 
                                      markeredgecolor='darkred', linewidth=1.5),
                        medianprops=dict(color='black', linewidth=2),
                        boxprops=dict(linewidth=1.5),
                        whiskerprops=dict(linewidth=1.5),
                        capprops=dict(linewidth=1.5))
        
        # Color the boxes
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        
        # Optionally add individual points
        if show_points:
            for i, x in enumerate(x_order):
                data = dataset_df[dataset_df['model_exp'] == x][metric].values
                # Add jitter to x-coordinates
                x_coords = np.random.normal(i + 1, 0.04, size=len(data))
                ax.scatter(x_coords, data, alpha=0.4, s=30, color='black', zorder=3)
        
        # Customize x-axis labels to show model names more clearly
        x_labels = []
        for label in x_order:
            model = label.split('\n')[0]
            exp_type_val = label.split('\n')[1]
            
            # Create short labels
            if exp_type_val == 'zeroshot':
                short_exp = 'ZS'
            elif 'finetune' in exp_type_val:
                short_exp = 'FT'
            elif exp_type_val == 'ttrlgc':
                short_exp = 'TTRL-GC'
            else:
                # Fallback: take first 4 chars uppercase or just use the name
                short_exp = exp_type_val[:4].upper() if len(exp_type_val) > 4 else exp_type_val
            
            x_labels.append(f"{model}\n({short_exp})")
        
        ax.set_xticklabels(x_labels, fontsize=9)
        
        # Add vertical lines to separate models
        for i in range(2, len(x_order), 2):
            ax.axvline(x=i + 0.5, color='gray', linestyle='--', alpha=0.3, linewidth=1)
        
        # Customize subplot
        ax.set_title(f'{data_name}', fontsize=12, fontweight='bold')
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        
        # Set y-axis limits if provided
        if ylim is not None:
            ax.set_ylim(ylim)
        
        # Only add y-label to leftmost subplot
        if idx == 0:
            ax.set_ylabel(metric, fontsize=12, fontweight='bold')
    
    # Create custom legend on the rightmost subplot
    legend_elements = [
        Patch(facecolor=palette.get(baseline_exp, '#8dd3c7'), alpha=0.7, label=baseline_exp),
        Patch(facecolor=palette.get(compare_exp, '#fb8072'), alpha=0.7, label=compare_exp),
        plt.Line2D([0], [0], marker='D', color='w', markerfacecolor='red', 
                   markersize=8, markeredgecolor='darkred', label='Mean')
    ]
    axes[-1].legend(handles=legend_elements, loc='upper right', fontsize=9)
    
    # Add overall title
    fig.suptitle(f'{metric} Comparison: {baseline_exp} vs {compare_exp} by Dataset', 
                fontsize=14, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    
    return fig, axes

In [ ]:
fig, ax = plot_boxplot_comparison_by_dataset(all_results_df, 
                                  metric='NMI',
                                  data_names = ['151509'],
                                  baseline_models=['Qwen2.5-7', 'gpt4o_mini'],
                                  compare_models=['Qwen2.5-7_CRF40', 'Qwen2.5-7_CRF70'],
                                  baseline_exp='zeroshot',
                                  compare_exp='ttrlgc',
                                  figsize=(12, 6),
                                  ylim=(0, 1),
                                  show_points=False)
plt.show()

In [ ]:
fig, ax = plot_boxplot_comparison_by_dataset(all_results_df, 
                                  metric='NMI',
                                  data_names = ['151509'],
                                  baseline_models=['Qwen2.5-7_CRF70'],
                                  compare_models=['Qwen2.5-7_CRF40'],
                                  baseline_exp='ttrlgc',
                                  compare_exp='ttrlgc',
                                  figsize=(12, 6),
                                  ylim=(0, 1),
                                  show_points=False)
plt.show()